In [ ]:
import numpy as np
import pandas as pd
import os
%reload_ext autoreload
%autoreload 2
%matplotlib inline
from fastai import *
from fastai.vision import *

In [ ]:
import fastai
print('The version of FastAI being used is:',fastai.__version__)

In [ ]:
img = open_image(Path('//kaggle/input/alzheimers-dataset-4-class-of-images/Alzheimer_s Dataset/train/VeryMildDemented/verymildDem972.jpg'))
print(img.shape)
img

In [ ]:
PATH = Path('/kaggle/input/alzheimers-dataset-4-class-of-images/Alzheimer_s Dataset/')


In [ ]:
transform = get_transforms(max_rotate=7.5,
                           max_zoom=1.15,
                           max_lighting=0.15,
                           max_warp=0.15,
                           p_affine=0.8, p_lighting = 0.8, 
                           xtra_tfms= [
                               pad(mode='zeros'),
                               symmetric_warp(magnitude=(-0.1,0.1)),
                               cutout(n_holes=(1,3), length=(5,5))
                           ])

In [ ]:
data = ImageDataBunch.from_folder(PATH, train="train/",
                                  test="test/",
                                  valid_pct=.4,
                                  ds_tfms=transform,
                                  size=112,bs=64, 
                                  ).normalize(imagenet_stats)

In [ ]:
data.show_batch(rows=3, figsize=(10,10))

In [ ]:
Category.__eq__ = lambda self, that: self.data == that.data
Category.__hash__ = lambda self: hash(self.obj)
Counter(data.train_ds.y)

In [ ]:
import torch.nn as nn

learn = cnn_learner(data, models.vgg16_bn, metrics=[FBeta(average='weighted'),accuracy], wd=1e-1, callback_fns=ShowGraph)

#learn = cnn_learner(data, models.vgg16_bn,loss_func=catagoricalcrossentropy , metrics=error_rate, wd=1e-1)#,pretrained=False)
learn.fit_one_cycle(4)

In [ ]:
Model_Path = Path('/kaggle/working/Alzheimer-stage-classifier-model/')
learn.model_dir = Model_Path
learn.save('checkpoint-1')

In [ ]:
learn.model
learn.recorder.plot_losses()

In [ ]:
learn.load('checkpoint-1');

In [ ]:
learn.unfreeze()
#learn.fit_one_cycle(8, max_lr=slice(1e-6,3e-4))
learn.fit_one_cycle(4, max_lr=slice(1e-6,3e-4))

In [ ]:
learn.model
learn.recorder.plot_losses()

In [ ]:
learn.save('checkpoint-2')

<a id="conclusions"></a>

 # Final Thoughts:

The goal of the Task was to hit an accuracy of 98% or greater, looks like the approach I chose to take just gets me to around 95% accuracy with an f_beta score of around .95 (Didn't set any random states so those number can be off by a percentage point or two when you rerun the notebook). The one nitpick about the model I have is that it will generate a small number of false negatives (the model would predict someone does not have alzheimers when they do).

In training and validation the model is able to get an accuracy score as high as 99% and an F_beta of around .98, however on test data the model is only able to achieve 95%. The final piece in getting this model's performance up is to focus on how to tone the overfitting down a smidge (In my personal experience, the best performing models are always slightly overfit). There is probably room for improvement with the transforms that I chose for the images. There may also be room for improvement by adding another resizing step or changing some of the dimensions in the current step. It's also possible that combining the results from this model with another would be necessary to close the gap for 98%.

In 3/21/2020, this was the most accurate model for this dataset that I'm aware of. Now this model isn't the most accurate, but the heatmap interptiability adds a lot of value to this project

This year I've challenged myself to complete one task on Kaggle per week, in order to develop a larger Data Science portfolio. If you found this notebook useful or interesting please give it an upvote. I'm always open to constructive feedback. If you have any questions, comments, concerns, or if you would like to collaborate on a future task of the week feel free to leave a comment here or message me directly. For past TOTW check out the link to my page on github for this ongoing project
https://github.com/Neil-Kloper/Weekly-Kaggle-Task/wiki

[Back to table of contents](#TOC) 

In [ ]:
learn.destroy()

In [ ]:
transform = get_transforms(max_rotate=7.5,
                           max_zoom=1.15,
                           max_lighting=0.15,
                           max_warp=0.15,
                           p_affine=0.8,
                           p_lighting = 0.8,
                           xtra_tfms= [
                               pad(mode='zeros'),
                               symmetric_warp(magnitude=(-0.1,0.1)),
                               cutout(n_holes=(1,6),length=(5,20))])

In [ ]:
data = ImageDataBunch.from_folder(PATH, train="train/",
#                                  valid="train/",
                                  test="test/",
                                  valid_pct=.2,
                                  ds_tfms=transform,
                                  size=224,bs=32, 
                                  ).normalize(imagenet_stats)

In [ ]:
learn = cnn_learner(data, models.vgg16_bn, pretrained=False, metrics=[error_rate, FBeta(average='weighted')], wd=1e-1, callback_fns=ShowGraph)
Model_Path = Path('/kaggle/working/Alzheimer-stage-classifier-model/')
learn.model_dir = Model_Path
learn.load('checkpoint-2');

In [ ]:
learn.lr_find()
learn.recorder.plot()

In [ ]:
learn.fit_one_cycle(17, max_lr=5e-4)

In [ ]:
learn.save('checkpoint-3')

In [ ]:
learn.model
learn.recorder.plot_losses()

In [ ]:
interp = ClassificationInterpretation.from_learner(learn)
losses,idxs = interp.top_losses()
len(data.valid_ds)==len(losses)==len(idxs)
interp.plot_confusion_matrix(figsize=(8,8))

In [ ]:
transform = get_transforms()

In [ ]:
def random_seed(seed_value, use_cuda):
    np.random.seed(seed_value) # cpu vars
    torch.manual_seed(seed_value) # cpu  vars
    random.seed(seed_value) # Python
    if use_cuda: 
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value) # gpu vars
        torch.backends.cudnn.deterministic = True  #needed
        torch.backends.cudnn.benchmark = False
random_seed(42, True)